# KAN-CDSCO2004U  Machine Learning and Deep Learning

## Lab 4 — Exercise 1: K-Nearest Neighbors (KNN)
**Estimated time: 30 minutes**

### Learning Objectives
By the end of this exercise, you will be able to:
- Understand how **KNN** works as a supervised classification algorithm
- Train a KNN model on the **MNIST** digit recognition dataset
- Evaluate model performance using **K-fold cross-validation**
- Use **GridSearchCV** to find the best hyperparameter **k**
- Interpret a **classification report** and **confusion matrix**

In this exercise, you will classify handwritten digits using **K-Nearest Neighbors**, one of the simplest supervised learning algorithms.

**How to work through this notebook:**
- 🏃 **RUN** cells = Just execute the code to see the output
- ✏️ **TODO** cells = Write your own code or answer questions
- 📖 **READ** cells = Explanations to help you understand the concepts

---
## Setup

🏃 **RUN** the cell below to import libraries.

In [ ]:
# Import needed libraries
# Author: Luca Gudi (lgg.digi@cbs.dk)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn.datasets import fetch_openml
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV

# Display options
plt.rcParams['figure.figsize'] = (10, 6)

---
## 1. Load and Explore the Data

📖 **READ**: The **MNIST** dataset is a classic benchmark in machine learning. It contains:
- **70,000** handwritten digit images (0–9)
- Each image is **28×28 pixels**, flattened to a **784-dimensional** vector

Since KNN is a **lazy learner** (it stores the entire training set and computes distances at prediction time), it can be slow on large datasets. To keep this exercise fast, we'll use a **subset of 5,000 samples**.

🏃 **RUN** the cells below to load and explore the data.

In [ ]:
# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Subsample for speed (KNN is slow on the full dataset!)
from sklearn.utils import resample
X_all, y_all = mnist["data"], mnist["target"]
X_sub, y_sub = resample(X_all, y_all, n_samples=5000, random_state=42, stratify=y_all)

print(f"Full dataset:   {X_all.shape[0]} samples, {X_all.shape[1]} features")
print(f"Subset used:    {X_sub.shape[0]} samples, {X_sub.shape[1]} features")
print(f"Labels:         {np.unique(y_sub)}")

In [ ]:
# Visualize some sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_sub[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"Label: {y_sub[i]}", fontsize=12)
    ax.axis('off')
plt.suptitle("Sample MNIST Digits", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Train/Test Split

📖 **READ**: Before training any model, we split the data into a **training set** (used to learn) and a **test set** (used to evaluate). A common split is **80% train / 20% test**.

✏️ **TODO**: Split the dataset into training and test sets.

In [ ]:
# TODO: Split the dataset into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_sub, y_sub, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

---
## 3. Basic KNN Classification

📖 **READ**:
KNN classifies a new data point by finding the **k nearest neighbors** in the training set and assigning the **majority class** among them.

Key concepts:
- **k** is the main hyperparameter — it controls how many neighbors vote
- **Small k** → low bias, high variance (risk of overfitting)
- **Large k** → high bias, low variance (risk of underfitting)
- Distance is typically measured using **Euclidean distance**

### 3.1 Create and Evaluate with Cross-Validation

✏️ **TODO**: Create a KNN classifier with `k=5` and evaluate it using **3-fold cross-validation**.

In [ ]:
# TODO: Create a KNN classifier with k=5
knn = KNeighborsClassifier(n_neighbors=5)

In [ ]:
# TODO: Evaluate using 3-fold cross-validation
scores = cross_val_score(knn, X_train, y_train, cv=3)

print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy:           {scores.mean():.4f}")
print(f"Standard deviation:      {scores.std():.4f}")

### 3.2 Fit and Predict

✏️ **TODO**: Fit the KNN model on the full training set, then make predictions on the test set.

In [ ]:
# TODO: Fit the model and make predictions
knn.fit(X_train, y_train)
pred = knn.predict(X_test)

print(f"Accuracy on test set: {metrics.accuracy_score(y_test, pred):.4f}")

---
## 4. Hyperparameter Tuning with Grid Search

📖 **READ**:
**GridSearchCV** automates the process of trying different hyperparameter values and selecting the best one. It combines:
- A **grid** of parameter values to try
- **Cross-validation** to evaluate each combination

This is how you find the **optimal k** without manually testing each value.

✏️ **TODO**: Use GridSearchCV to find the best `k` from `[3, 5, 7]`.

In [ ]:
# TODO: Set up the parameter grid and run GridSearchCV
param_grid = {'n_neighbors': [3, 5, 7]}

grid_search = GridSearchCV(
    KNeighborsClassifier(), param_grid, cv=3, return_train_score=True
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score:   {grid_search.best_score_:.4f}")

---
## 5. Final Evaluation

✏️ **TODO**: Use the best model from GridSearchCV to make predictions on the test set, then display the **classification report** and **confusion matrix**.

In [ ]:
# TODO: Predict on the test set using the best model
best_pred = grid_search.predict(X_test)

print("Classification Report:")
print(metrics.classification_report(y_test, best_pred))

In [ ]:
# TODO: Display the confusion matrix
cm = metrics.confusion_matrix(y_test, best_pred)
disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix — KNN on MNIST")
plt.show()

### ✏️ TODO: Answer the following questions

**Q1: What does the `k` parameter control in KNN? Why is choosing the right `k` important?**

Your answer: The `k` parameter controls how many neighbors are used to vote on the classification of a new data point. A small `k` can lead to overfitting (high variance), while a large `k` can lead to underfitting (high bias). Cross-validation and grid search help us find the right balance.

**Q2: What are the main advantages and disadvantages of KNN?**

| | Advantages | Disadvantages |
| :--- | :--- | :--- |
| **1** | Simple and easy to implement | Does not scale well to large datasets |
| **2** | No training phase (lazy learner) | Sensitive to irrelevant features |
| **3** | Few hyperparameters to tune | Prone to the curse of dimensionality |

---
## Summary

In this lab, you learned how to:

| Section | Technique | Python Code / sklearn Class |
| :--- | :--- | :--- |
| **2. Train/Test Split** | Splitting data for evaluation | `train_test_split(X, y, test_size=0.2)` |
| **3.1 Cross-Validation** | K-fold evaluation of a model | `cross_val_score(model, X, y, cv=3)` |
| **3.2 Basic KNN** | Classify digits using k neighbors | `KNeighborsClassifier(n_neighbors=k)` |
| **4. Grid Search** | Automated hyperparameter tuning | `GridSearchCV(model, param_grid, cv=3)` |
| **5. Evaluation** | Assess predictions in detail | `classification_report()`, `confusion_matrix()` |

**Key takeaways:**
- **KNN** is a simple, non-parametric classifier that uses distance to classify new points
- **Cross-validation** gives a more reliable estimate of model performance than a single train/test split
- **GridSearchCV** automates finding the best hyperparameters
- KNN **does not scale well** — as the dataset grows, prediction becomes increasingly slow